# stress_level_sample: sample grain, quality, cadence and relationships
For the internship presentation on Friday October 23, 2026. Only this sample table and its required comparison table are read. SQL returns aggregate distributions and period summaries; raw sample rows are not loaded into pandas. All queries use one read-only repeatable-read snapshot.

One candidate row is a participant's recorded `stress_level` at `sample_point_sec`, within a labelled export/reporting period. `additional_id` can repeat across samples and is not assumed to identify one sample. IDs/group/type identify context; calendar/week/day label records; start/end/duration describe reporting windows.

Score definition, range, sentinel codes and algorithms are unverified: no source dictionary is supplied. Timestamp names suggest Unix seconds, but conversions and arithmetic are diagnostics. Negative values, zeros and nulls are reported separately, not assigned invented meanings. No medical thresholds, diagnosis, causation or model validation are used.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from sqlalchemy import create_engine, inspect, text
from dotenv import dotenv_values
from IPython.display import display, JSON, Markdown
PROJECT=Path.cwd()
url=dotenv_values(PROJECT/'.env').get('DATABASE_URL')
assert url, 'Configure DATABASE_URL privately in .env.'
engine=create_engine(url); del url
connection=engine.connect().execution_options(isolation_level='REPEATABLE READ')
transaction=connection.begin()
connection.execute(text('SET TRANSACTION READ ONLY'))
connection.execute(text("SET LOCAL statement_timeout = '240s'"))
def query(sql): return pd.read_sql(text(sql),connection)
key=['sdv_user_id','calendar_date','start_measure_period_sec','end_measure_period_sec']
results={}
def valid_key(df,k): return not df[k].isna().any(axis=None) and not df.duplicated(k).any()
def show_schema(table):
    cols=inspect(connection).get_columns(table)
    display(pd.DataFrame([{'column':c['name'],'SQL_type':str(c['type'])} for c in cols]))
    return [c['name'] for c in cols]
def audit_keys(df,candidates):
    display(pd.DataFrame([{'key':name,'null_key_rows':int(df[k].isna().any(axis=1).sum()),
      'rows_in_duplicate_keys':int(df.duplicated(k,keep=False).sum()),'unique_nonnull':valid_key(df,k)} for name,k in candidates.items()]))
def period_audit(p):
    p=p.copy(); p['calendar_date']=pd.to_datetime(p.calendar_date,errors='coerce')
    elapsed=p.end_measure_period_sec-p.start_measure_period_sec
    ordered=p.sort_values(key)
    prior=ordered.groupby('sdv_user_id').end_measure_period_sec.transform(lambda x:x.cummax().shift())
    utc=pd.to_datetime(p.start_measure_period_sec,unit='s',utc=True,errors='coerce')
    stats={'periods':len(p),'participants':p.sdv_user_id.nunique(),
      'null_key_rows':int(p[key].isna().any(axis=1).sum()),'duplicate_period_keys':int(p.duplicated(key,keep=False).sum()),
      'nonpositive_periods':int(elapsed.le(0).sum()),'exactly_24_hour_periods':int(elapsed.eq(86400).sum()),
      'overlapping_periods':int(ordered.start_measure_period_sec.lt(prior).sum()),
      'UTC_start_date_matches_label':int(utc.dt.tz_localize(None).dt.normalize().eq(p.calendar_date).sum())}
    if 'duration_measure_period_sec' in p:
        stats['duration_equals_end_minus_start']=int(p.duration_measure_period_sec.eq(elapsed).sum())
    display(pd.DataFrame([stats])); display((elapsed/3600).value_counts().sort_index().rename_axis('period_hours').to_frame('periods'))
    coverage=p.groupby('sdv_user_id').agg(first=('calendar_date','min'),last=('calendar_date','max'),observed_dates=('calendar_date','nunique'),periods=('calendar_date','size'))
    coverage['span_days']=(coverage['last']-coverage['first']).dt.days+1
    coverage['missing_dates_within_span']=coverage.span_days-coverage.observed_dates
    coverage['observed_date_percent']=100*coverage.observed_dates/coverage.span_days
    display(coverage[['observed_dates','periods','span_days','missing_dates_within_span','observed_date_percent']].describe().T)
    stats.update({'min_observed_dates_per_participant':int(coverage.observed_dates.min()),
      'max_observed_dates_per_participant':int(coverage.observed_dates.max()),
      'min_observed_date_percent':coverage.observed_date_percent.min(),'max_observed_date_percent':coverage.observed_date_percent.max()})
    return stats
def checked_join(left,right,label):
    if not (valid_key(left,key) and valid_key(right,key)):
        print(label+': skipped; keys are not unique and non-null.'); return None
    joined=left.merge(right,on=key,how='outer',suffixes=('_left','_right'),indicator=True,validate='one_to_one')
    counts={str(k):int(v) for k,v in joined._merge.value_counts().items()}
    results[label+'_period_join']=counts
    results[label+'_participant_overlap']={'left':left.sdv_user_id.nunique(),'right':right.sdv_user_id.nunique(),
      'shared':len(set(left.sdv_user_id.dropna()) & set(right.sdv_user_id.dropna()))}
    display(pd.Series(counts,name='periods').rename_axis(label+'_status').to_frame())
    return joined

TABLE='stress_level_sample'
VALUE='stress_level'
columns=show_schema(TABLE)


## 1. Inventory, all-column nulls and recorded-value distribution
Count raw rows, participants, record IDs and dates. Report all score values with frequencies, including negative/zero candidates. Nonnegative subsets are operational sensitivity checks, not certified valid measurements. Samples outside bounds and exactly at endpoints are counted separately. SQL aggregation avoids a large raw-data display.

In [ ]:
inventory=query("""SELECT COUNT(*) rows,COUNT(DISTINCT sdv_user_id) participants,
 COUNT(DISTINCT additional_id) record_ids,COUNT(DISTINCT calendar_date) labelled_dates,
 COUNT(*) FILTER(WHERE stress_level IS NULL) null_values,COUNT(*) FILTER(WHERE stress_level=0) zero_values,
 COUNT(*) FILTER(WHERE stress_level<0) negative_values,MIN(stress_level) min_value,MAX(stress_level) max_value,
 COUNT(*) FILTER(WHERE sample_point_sec<start_measure_period_sec) before_start,
 COUNT(*) FILTER(WHERE sample_point_sec>end_measure_period_sec) after_end,
 COUNT(*) FILTER(WHERE sample_point_sec=end_measure_period_sec) exactly_at_end,
 COUNT(*) FILTER(WHERE (to_timestamp(sample_point_sec) AT TIME ZONE 'UTC')::date=calendar_date) UTC_sample_date_matches_label,
 COUNT(*) FILTER(WHERE end_measure_period_sec-start_measure_period_sec<>duration_measure_period_sec) duration_mismatch_rows
 FROM stress_level_sample""")
display(inventory); results['inventory']=inventory.iloc[0].to_dict()
null_sql=', '.join('COUNT(*) FILTER(WHERE "'+c+'" IS NULL) AS "'+c+'"' for c in columns)
nulls=query('SELECT '+null_sql+' FROM '+TABLE).T.rename(columns={0:'null_rows'})
display(nulls); results['null_cells']=int(nulls.null_rows.sum())
distribution=query('SELECT '+VALUE+' value,COUNT(*) records FROM '+TABLE+' GROUP BY '+VALUE+' ORDER BY '+VALUE+' NULLS FIRST')
display(distribution)
results['nonpositive_values']=distribution.loc[distribution.value.le(0)].to_dict('records')
fig,ax=plt.subplots(figsize=(9,3.5)); d=distribution.dropna(subset=['value']); ax.bar(d.value,d.records)
ax.set(title='All recorded sample values',xlabel='Recorded score; semantics unverified',ylabel='Sample rows')
plt.tight_layout(); plt.show()


## 2. Raw keys, exact duplicates and period aggregation
Test participant/date/start/end/timestamp and participant/timestamp keys separately. Exact duplicates include all raw columns; same timestamp with different scores is a conflict. Shared timestamps across reporting periods flag overlap/export duplication risk, not necessarily identical full rows.

Return one aggregate per exact participant/date/start/end period and check metadata variation before joining. Keep all-value, nonnegative and timestamp-collapsed nonnegative means distinct. Timestamp collapse is used only for sensitivity, excludes conflicting timestamps, and does not modify source data or remove cross-period repetition.

In [ ]:
dup=query("""WITH d AS(SELECT sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec,sample_point_sec,
 COUNT(*) n,COUNT(DISTINCT stress_level) values_n,
 COUNT(DISTINCT (additional_id,sdv_group_id,sdv_data_type,duration_measure_period_sec,week_nr,week_day,stress_level)) identities
 FROM stress_level_sample GROUP BY sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec,sample_point_sec)
 SELECT COUNT(*) period_timestamp_keys,COUNT(*) FILTER(WHERE n>1) repeated_period_timestamp_keys,
 COALESCE(SUM(n-1),0) rows_beyond_period_timestamp_key,COUNT(*) FILTER(WHERE values_n>1) conflicting_period_timestamp_keys,
 COALESCE(SUM(n-identities),0) exact_duplicate_rows_after_first FROM d""")
display(dup); results['duplicates']=dup.iloc[0].to_dict()
cross=query("""WITH d AS(SELECT sdv_user_id,sample_point_sec,COUNT(*) n,
 COUNT(DISTINCT (calendar_date,start_measure_period_sec,end_measure_period_sec)) periods,COUNT(DISTINCT stress_level) values_n
 FROM stress_level_sample GROUP BY sdv_user_id,sample_point_sec)
 SELECT COUNT(*) participant_timestamp_keys,COUNT(*) FILTER(WHERE n>1) repeated_participant_timestamp_keys,
 COUNT(*) FILTER(WHERE periods>1) timestamps_shared_across_periods,
 COUNT(*) FILTER(WHERE values_n>1) conflicting_participant_timestamp_keys FROM d""")
display(cross); results['global_timestamp_keys']=cross.iloc[0].to_dict()
p=query("""SELECT sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec,
 COUNT(*) sample_rows,COUNT(DISTINCT sample_point_sec) distinct_timestamps,
 COUNT(DISTINCT additional_id) record_variants,MIN(additional_id) additional_id,
 COUNT(DISTINCT sdv_group_id) group_variants,MIN(sdv_group_id) sdv_group_id,
 COUNT(DISTINCT duration_measure_period_sec) duration_variants,MIN(duration_measure_period_sec) duration_measure_period_sec,
 MIN(sample_point_sec) first_sample_sec,MAX(sample_point_sec) last_sample_sec,
 MIN(stress_level) min_all,MAX(stress_level) max_all,AVG(stress_level) mean_all,
 MAX(stress_level) FILTER(WHERE stress_level>=0) max_nonnegative,AVG(stress_level) FILTER(WHERE stress_level>=0) mean_nonnegative,
 COUNT(*) FILTER(WHERE stress_level>=0) nonnegative_samples
 FROM stress_level_sample GROUP BY sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec""")
p['calendar_date']=pd.to_datetime(p.calendar_date,errors='coerce')
dedup=query("""WITH d AS(SELECT sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec,sample_point_sec,
 MIN(stress_level) value,COUNT(DISTINCT stress_level) values_n FROM stress_level_sample
 GROUP BY sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec,sample_point_sec)
 SELECT sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec,
 AVG(value) FILTER(WHERE values_n=1 AND value>=0) mean_unique_timestamp_nonnegative,
 COUNT(*) FILTER(WHERE values_n>1) conflicting_timestamps FROM d
 GROUP BY sdv_user_id,calendar_date,start_measure_period_sec,end_measure_period_sec""")
dedup['calendar_date']=pd.to_datetime(dedup.calendar_date,errors='coerce')
p=p.merge(dedup,on=key,validate='one_to_one')
audit_keys(p,{'derived_period':key,'participant_date':['sdv_user_id','calendar_date'],'record_id':['additional_id']})
results['ambiguous_period_metadata']=int((p[['record_variants','group_variants','duration_variants']].ne(1).any(axis=1)).sum())
display(p[['sample_rows','distinct_timestamps','nonnegative_samples']].describe().T)
results['samples_per_period']={'minimum':p.sample_rows.min(),'median':p.sample_rows.median(),'maximum':p.sample_rows.max()}
results['period_mean_weighting_difference_max']=(p.mean_nonnegative-p.mean_unique_timestamp_nonnegative).abs().max()


## 3. Reporting coverage and cadence
Reporting windows are checked separately from actual observations. End minus start, nonpositive windows, exact 24-hour windows and overlaps are diagnostics, not proof of wear or complete days. UTC conversion assumes Unix seconds only as a plausibility check; local calendar labels and inclusive/exclusive endpoints need documentation. Date coverage is unique observed dates divided by dates in each participant's first-to-last observed span; enrolment boundaries are unknown. Missing dates are never zero values. Participant identifiers are retained only in memory for grouping and are not displayed.

For within-export cadence, collapse timestamps only for the spacing diagnostic and partition by participant, record ID, date and bounds. The mode/median are observed intervals, not a nominal schedule. Across-record **boundary** gaps below compare the next record's first sample with the latest preceding record's last sample. They are record-boundary diagnostics, not the participant-wide consecutive-sample sequence; negative gaps indicate overlap. Neither long gaps nor record counts prove wear/nonwear.

In [ ]:
results['periods']=period_audit(p)
gaps=query("""WITH t AS(SELECT sdv_user_id,additional_id,calendar_date,start_measure_period_sec,end_measure_period_sec,sample_point_sec
 FROM stress_level_sample WHERE sample_point_sec IS NOT NULL GROUP BY sdv_user_id,additional_id,calendar_date,start_measure_period_sec,end_measure_period_sec,sample_point_sec),
 g AS(SELECT sample_point_sec-LAG(sample_point_sec) OVER(PARTITION BY sdv_user_id,additional_id,calendar_date,start_measure_period_sec,end_measure_period_sec ORDER BY sample_point_sec) gap_sec FROM t)
 SELECT COUNT(*) intervals,MIN(gap_sec) min_sec,MAX(gap_sec) max_sec,
 MODE() WITHIN GROUP(ORDER BY gap_sec) observed_mode_sec,
 PERCENTILE_CONT(.5) WITHIN GROUP(ORDER BY gap_sec) median_sec,
 PERCENTILE_CONT(.9) WITHIN GROUP(ORDER BY gap_sec) p90_sec,
 PERCENTILE_CONT(.99) WITHIN GROUP(ORDER BY gap_sec) p99_sec,
 COUNT(*) FILTER(WHERE gap_sec>3600) gaps_over_one_hour FROM g WHERE gap_sec IS NOT NULL""")
display(gaps); results['within_record_cadence']=gaps.iloc[0].to_dict()
ordered=p.sort_values(['sdv_user_id','first_sample_sec','last_sample_sec'])
prior_last=ordered.groupby('sdv_user_id').last_sample_sec.transform(lambda x:x.cummax().shift())
boundary=ordered.first_sample_sec-prior_last
display(boundary.describe().to_frame('record_boundary_gap_seconds'))
results['record_boundary_gaps']={'count':int(boundary.notna().sum()),'negative_overlap':int(boundary.lt(0).sum()),
 'above_one_hour':int(boundary.gt(3600).sum()),'maximum_sec':boundary.max()}
span=(p.last_sample_sec-p.first_sample_sec)/(p.end_measure_period_sec-p.start_measure_period_sec)
display(span.describe().to_frame('first_to_last_sample_span_divided_by_window'))
print('The span fraction is not a valid-sample coverage or wear-time fraction.')
fig,axes=plt.subplots(1,2,figsize=(11,3.5))
p.sample_rows.plot.hist(bins=25,ax=axes[0]); axes[0].set(title='Samples per reporting window',xlabel='Sample rows',ylabel='Periods')
span.plot.hist(bins=25,ax=axes[1]); axes[1].set(title='Sample span / reporting window',xlabel='Span fraction (not wear time)',ylabel='Periods')
plt.tight_layout(); plt.show()


## 4. Validated relationship with stress summaries
Read only the small stress summary table. Outer one-to-one period joins preserve unmatched keys; require internally unique metadata and agreeing ID/group/duration context before value comparison. Compare sample max with reported max, and row-weighted all/nonnegative/timestamp-collapsed means with the nonnegative reported average. These are alternative operational calculations, not asserted source algorithms. Negative samples and duplicate weighting may change results, so exclusions and differences are visible. Do not reconstruct duration categories by assigning undocumented score thresholds.

In [ ]:
summary=query('SELECT * FROM stress'); summary['calendar_date']=pd.to_datetime(summary.calendar_date,errors='coerce')
audit_keys(summary,{'period':key,'participant_date':['sdv_user_id','calendar_date']})
j=checked_join(p,summary,'stress_summary')
comparisons=[]
if j is not None:
    both=j._merge.eq('both')
    use=both & j.record_variants.eq(1) & j.group_variants.eq(1) & j.duration_variants.eq(1)
    for field in ['additional_id','sdv_group_id','duration_measure_period_sec']:
        equal=j[field+'_left'].eq(j[field+'_right']); results['context_'+field+'_equal']=int((use & equal).sum()); use &= equal
    results['comparable_context_periods']=int(use.sum())
    for sample_metric,summary_metric in [('max_all','max_stress_level'),('max_nonnegative','max_stress_level'),
       ('mean_all','average_stress_level'),('mean_nonnegative','average_stress_level'),('mean_unique_timestamp_nonnegative','average_stress_level')]:
        complete=use & j[sample_metric].notna() & j[summary_metric].notna() & j[summary_metric].ge(0)
        delta=j.loc[complete,sample_metric]-j.loc[complete,summary_metric]
        comparisons.append({'sample_calculation':sample_metric,'summary_field':summary_metric,'complete_pairs':int(complete.sum()),
          'excluded_matched':int((use & ~complete).sum()),'exact_equal':int(delta.eq(0).sum()),'within_absolute_1':int(delta.abs().le(1).sum()),
          'median_absolute_difference':delta.abs().median(),'maximum_absolute_difference':delta.abs().max()})
    display(pd.DataFrame(comparisons)); results['comparisons']=comparisons
    d=j.loc[use & j.mean_nonnegative.notna() & j.average_stress_level.ge(0)]
    fig,axes=plt.subplots(1,2,figsize=(11,3.5))
    axes[0].scatter(d.average_stress_level,d.mean_nonnegative,s=15,alpha=.4)
    axes[0].set(title='Nonnegative sample mean vs summary',xlabel='Summary average',ylabel='Row-weighted sample mean')
    (d.mean_nonnegative-d.average_stress_level).plot.hist(bins=25,ax=axes[1]); axes[1].set(title='Sample mean minus summary',xlabel='Recorded-unit difference',ylabel='Matched complete periods')
    plt.tight_layout(); plt.show()


## Verified findings and teacher takeaway

- **460,975 rows, 981 periods, eight participants**; all fields are non-null, but **147,538 scores are negative**: 104,279 values of −1 and 43,259 of −2. No zero values occur. These are potential sentinel codes with unverified meanings.
- **108 exact duplicate rows** occur across 74 repeated within-period timestamp keys, with no within-period value conflicts. Participant/timestamp alone is not unique: 164 timestamps span periods, and **one has conflicting scores across periods**. No global deduplication or conflict resolution is assumed.
- All timestamps lie within reporting bounds, and duration equals end minus start. There are **962 exactly-24-hour periods**, 19 other-duration periods and **four overlapping windows**. Observed dates per participant range 16–250; within observed spans, date coverage ranges 89.36%–100%, not wear coverage.
- Observed within-record mode/median/p90/p99 are **180 seconds**, with **29 gaps above one hour** and a maximum 55,440 seconds. Across-record boundary diagnostics are separate: 19 exceed one hour and two are negative overlaps. These observations do not prove nominal cadence or nonwear.
- Exact-period stress-summary join: **978 matches, three sample-only, nine summary-only**; ID/group/duration agree for every match. Only **90/948** nonnegative sample-max/summary-max pairs are exactly equal; median absolute difference is 4 recorded units.
- Including negative scores yields sample-mean/summary-average median absolute difference **8.146** across 951 complete pairs. The operational nonnegative subset gives **0.383** across 948 pairs, with **857/948 within one unit** and maximum difference **17.211**. Timestamp-collapsed sensitivity gives median difference 0.384. Duplicate weighting changes a period's nonnegative mean by at most 0.633 in this snapshot. Neither filtering nor arithmetic agreement verifies a device algorithm.

**Takeaway:** “Stress samples contain undocumented negative codes and some duplicate exports. I separated those issues, checked actual cadence and matched periods before comparing with summaries. Filter choice changes the apparent agreement substantially, so the source definitions are essential.”

No medical, causal or predictive interpretation is made. Missing periods are never treated as zero.


## Teacher-facing method and limits
“I separated individual samples from export/reporting periods, counted invalidity candidates and duplicate timestamp risks, described observed spacing, then matched period aggregates using validated keys.”

Read each comparison with its denominator. Operational score filtering does not certify validity. Duplicate exports, irregular gaps, missing periods and unequal follow-up can affect sample-weighted summaries. First-to-last spans are not wear time. The source algorithms, score meanings, timestamp conventions and actual wear remain unresolved. No medical, causal or predictive conclusions follow. CSV, RAG and model building remain deferred.

In [ ]:
# End the read-only snapshot. No database changes were made.
transaction.rollback(); connection.close(); engine.dispose()
def scalar(v):
    if hasattr(v,'item'): return v.item()
    return str(v)
safe_results=json.loads(json.dumps(results,default=scalar))
display(JSON(safe_results))
print('All analysis cells completed; read-only connection closed.')
